# Исследование и обучение модели: Heart Attack Risk

Этот ноутбук содержит:
- загрузку и первичный анализ данных;
- очистку и предобработку;
- сравнение baseline-модели и CatBoost;
- подбор порога классификации;
- обучение финальной модели;
- сохранение модели и файла `predictions.csv`.

> В ноутбуке только исследования и эксперименты. Код приложения затем выносится в отдельные модули проекта.


## 1. Импорты и настройки

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

RANDOM_STATE = 42
TARGET = "Heart Attack Risk (Binary)"
ID_COL = "id"
DROP_COLS = ["Unnamed: 0"]


## 2. Загрузка данных

Если ноутбук лежит в `notebooks/`, а данные в `data/`, оставь относительные пути как ниже.
Если запускаешь файл из другой папки, поправь пути.


In [ ]:
TRAIN_PATH = Path("../data/heart_train.csv")
TEST_PATH = Path("../data/heart_test.csv")

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("train shape:", train.shape)
print("test shape:", test.shape)

train.head()


## 3. Первичный анализ данных

In [ ]:
print("Колонки train:")
print(train.columns.tolist())
print()

print("Типы данных:")
display(train.dtypes)

print("Пропуски в train:")
display(train.isna().sum().sort_values(ascending=False).head(20))

print("Пропуски в test:")
display(test.isna().sum().sort_values(ascending=False).head(20))


In [ ]:
print("Распределение таргета:")
display(train[TARGET].value_counts())
display((train[TARGET].value_counts(normalize=True) * 100).round(2))


In [ ]:
print("Проверка дублей в train:", train.duplicated().sum())
print("Проверка дублей в test:", test.duplicated().sum())

print("Уникальные значения Gender:")
display(train["Gender"].value_counts(dropna=False))


### Выводы после первичного анализа

1. `Unnamed: 0` выглядит как технический индекс — его нужно удалить.
2. `id` нужен для финального файла предсказаний, но не должен участвовать в обучении.
3. В `Gender` есть загрязнение значений (`Male`, `Female`, `1.0`, `0.0`) — нужно привести к единому виду.
4. В ряде бинарных колонок есть пропуски — их можно заполнить модой.
5. Классы умеренно несбалансированы, поэтому основной метрикой возьмём **ROC-AUC**, а дополнительно будем смотреть **F1** и **Recall**.


## 4. Очистка и подготовка данных

In [ ]:
def clean_gender(series: pd.Series) -> pd.Series:
    mapping = {
        "1": "Male",
        "1.0": "Male",
        1: "Male",
        1.0: "Male",
        "0": "Female",
        "0.0": "Female",
        0: "Female",
        0.0: "Female",
        "male": "Male",
        "female": "Female",
        "Male": "Male",
        "Female": "Female",
    }
    return series.map(lambda x: mapping.get(x, x))


def basic_clean(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result = result.drop(columns=[col for col in DROP_COLS if col in result.columns], errors="ignore")
    if "Gender" in result.columns:
        result["Gender"] = clean_gender(result["Gender"]).astype("object")
    return result


train_clean = basic_clean(train)
test_clean = basic_clean(test)

train_clean.head()


In [ ]:
X = train_clean.drop(columns=[TARGET])
y = train_clean[TARGET].astype(int)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)


In [ ]:
categorical_features = ["Gender"]
numeric_features = [col for col in X_train.columns if col not in categorical_features + [ID_COL]]

print("categorical_features:", categorical_features)
print("numeric_features count:", len(numeric_features))


## 5. Baseline-модель: Logistic Regression

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

baseline_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

baseline_model = Pipeline(
    steps=[
        ("preprocessor", baseline_preprocessor),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]
)

baseline_model.fit(X_train.drop(columns=[ID_COL]), y_train)
baseline_proba = baseline_model.predict_proba(X_valid.drop(columns=[ID_COL]))[:, 1]
baseline_pred = (baseline_proba >= 0.5).astype(int)

baseline_metrics = {
    "accuracy": accuracy_score(y_valid, baseline_pred),
    "precision": precision_score(y_valid, baseline_pred, zero_division=0),
    "recall": recall_score(y_valid, baseline_pred, zero_division=0),
    "f1": f1_score(y_valid, baseline_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_valid, baseline_proba),
}

pd.Series(baseline_metrics).sort_index()


## 6. Основная модель: CatBoost

CatBoost хорошо подходит для табличных данных и умеет работать с категориальными признаками и пропусками.


In [ ]:
catboost_features = [col for col in X_train.columns if col != ID_COL]
catboost_cat_features = [col for col in categorical_features if col in catboost_features]

X_train_cb = X_train[catboost_features].copy()
X_valid_cb = X_valid[catboost_features].copy()

cat_model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=False,
)

cat_model.fit(
    X_train_cb,
    y_train,
    cat_features=catboost_cat_features,
    eval_set=(X_valid_cb, y_valid),
    use_best_model=True,
)

cat_proba = cat_model.predict_proba(X_valid_cb)[:, 1]
cat_pred_05 = (cat_proba >= 0.5).astype(int)

cat_metrics_05 = {
    "accuracy": accuracy_score(y_valid, cat_pred_05),
    "precision": precision_score(y_valid, cat_pred_05, zero_division=0),
    "recall": recall_score(y_valid, cat_pred_05, zero_division=0),
    "f1": f1_score(y_valid, cat_pred_05, zero_division=0),
    "roc_auc": roc_auc_score(y_valid, cat_proba),
}

pd.Series(cat_metrics_05).sort_index()


## 7. Подбор оптимального threshold по F1

In [ ]:
threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.01):
    pred = (cat_proba >= threshold).astype(int)
    threshold_results.append(
        {
            "threshold": round(float(threshold), 2),
            "precision": precision_score(y_valid, pred, zero_division=0),
            "recall": recall_score(y_valid, pred, zero_division=0),
            "f1": f1_score(y_valid, pred, zero_division=0),
        }
    )

threshold_df = pd.DataFrame(threshold_results).sort_values("f1", ascending=False)
threshold_df.head(10)


In [ ]:
best_threshold = float(threshold_df.iloc[0]["threshold"])
print("Best threshold:", best_threshold)

best_pred = (cat_proba >= best_threshold).astype(int)

best_metrics = {
    "accuracy": accuracy_score(y_valid, best_pred),
    "precision": precision_score(y_valid, best_pred, zero_division=0),
    "recall": recall_score(y_valid, best_pred, zero_division=0),
    "f1": f1_score(y_valid, best_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_valid, cat_proba),
}

print("Confusion matrix:")
print(confusion_matrix(y_valid, best_pred))
print()
print("Classification report:")
print(classification_report(y_valid, best_pred, zero_division=0))
print()
pd.Series(best_metrics).sort_index()


## 8. Сравнение моделей

In [ ]:
comparison = pd.DataFrame(
    [
        {"model": "LogisticRegression (threshold=0.5)", **baseline_metrics},
        {"model": "CatBoost (threshold=0.5)", **cat_metrics_05},
        {"model": f"CatBoost (threshold={best_threshold})", **best_metrics},
    ]
)

comparison.sort_values("roc_auc", ascending=False)


### Вывод по моделям

- `LogisticRegression` — baseline.
- `CatBoost` — основная модель.
- Для финального бинарного ответа используется порог, подобранный по F1 на валидации.
- В качестве основной метрики для обоснования в проекте используется **ROC-AUC**.


## 9. Обучение финальной модели на всём train

In [ ]:
X_full = train_clean.drop(columns=[TARGET]).copy()
y_full = train_clean[TARGET].astype(int)

final_features = [col for col in X_full.columns if col != ID_COL]
X_full_cb = X_full[final_features].copy()
X_test_cb = test_clean[final_features].copy()

final_model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=False,
)

final_model.fit(
    X_full_cb,
    y_full,
    cat_features=catboost_cat_features,
)


## 10. Предсказание на test и сохранение `predictions.csv`

In [ ]:
test_proba = final_model.predict_proba(X_test_cb)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

predictions = pd.DataFrame(
    {
        "id": test_clean[ID_COL].values,
        "prediction": test_pred.astype(int),
    }
)

predictions.head()


In [ ]:
OUTPUT_DIR = Path("../data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_PATH = OUTPUT_DIR / "predictions.csv"
predictions.to_csv(PREDICTIONS_PATH, index=False)

print("Файл сохранён:", PREDICTIONS_PATH)
print("Размер predictions:", predictions.shape)


## 11. Сохранение артефактов модели

Ниже — простой вариант сохранения, чтобы потом использовать модель в приложении.


In [ ]:
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "model.cbm"
META_PATH = MODELS_DIR / "metadata.json"

final_model.save_model(MODEL_PATH)

metadata = {
    "target": TARGET,
    "id_column": ID_COL,
    "drop_columns": DROP_COLS,
    "categorical_columns": categorical_features,
    "final_features": final_features,
    "best_threshold": best_threshold,
    "validation_metrics": best_metrics,
}

with open(META_PATH, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=4)

print("Model saved to:", MODEL_PATH)
print("Metadata saved to:", META_PATH)


## 12. Финальные выводы

В ходе исследования были выявлены:
- технический признак `Unnamed: 0`;
- служебный идентификатор `id`, который не должен использоваться в обучении;
- загрязнение категориального признака `Gender`;
- пропуски в части признаков.

После очистки данных были обучены baseline-модель и CatBoost. В качестве основной метрики выбрана **ROC-AUC**, так как она лучше подходит для бинарной классификации с умеренным дисбалансом классов и оценивает качество ранжирования объектов по вероятности риска. Для итогового бинарного решения дополнительно подбирался порог классификации по **F1-score**.

Для финального решения используется **CatBoostClassifier**, обученный на всех доступных данных train, а результат для тестовой выборки сохраняется в файл `predictions.csv` в формате:
- `id`
- `prediction`
